# 🔬 e-K Cha Label? — Run on Google Colab

This notebook installs everything, writes the app files, and launches the Streamlit app with a public URL.

**Steps:**
1. Run Cell 1 (install + write files) — takes ~2-3 min first time
2. Run Cell 2 (launch app) — gives you a public URL to open
3. Upload your label image in the app and analyze!

In [ ]:
#@title 📦 Cell 1: Install dependencies + write app files
!pip install -q streamlit pillow numpy pandas plotly 'surya-ocr>=0.16,<0.17' localtunnel 2>&1 | tail -5

import os
os.makedirs('/content/ekchalabel', exist_ok=True)

# Download the dataset and app files from your GitHub repo
# Option A: If your repo is public, clone it:
# !git clone https://github.com/YOUR_USERNAME/ekchalabel /content/ekchalabel

# Option B: Upload files manually:
# Upload utils.py, app.py, eatsafe_master_database.csv to /content/ekchalabel/
# using the Colab file browser (folder icon on the left)

print('\n✅ Dependencies installed!')
print('📂 Now upload utils.py, app.py, and eatsafe_master_database.csv')
print('   to /content/ekchalabel/ using the file browser on the left.')
print('   Or uncomment the git clone line above if your repo is public.')

In [ ]:
#@title 🚀 Cell 2: Launch Streamlit app with public URL
import subprocess, threading, time, requests

# Set environment for Surya
os.environ['TORCH_DEVICE'] = 'cpu'  # Change to 'cuda' if you have GPU runtime
os.environ['RECOGNITION_BATCH_SIZE'] = '8'
os.environ['DETECTOR_BATCH_SIZE'] = '4'

# Check files exist
app_dir = '/content/ekchalabel'
for f in ['app.py', 'utils.py', 'eatsafe_master_database.csv']:
    assert os.path.exists(os.path.join(app_dir, f)), f'❌ Missing {f} in {app_dir}'
print('✅ All files found')

# Start Streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.fileWatcherType', 'none',
     '--browser.gatherUsageStats', 'false'],
    cwd=app_dir,
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for it to start
print('⏳ Starting Streamlit...')
for _ in range(30):
    time.sleep(1)
    try:
        r = requests.get('http://localhost:8501', timeout=2)
        if r.status_code == 200: break
    except: pass
print('✅ Streamlit is running')

# Start localtunnel for public URL
tunnel = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)
output = tunnel.stdout.readline().decode('utf-8')
if 'url' in output.lower() or 'https' in output:
    print(f'\n🌐 Your app is live at: {output.strip()}')
    print('   (You may need to click "Click to Continue" on the tunnel page)')
else:
    # Fallback: try to extract URL
    print(f'\n📡 Tunnel output: {output.strip()}')
    print('   If no URL appeared, try running this cell again.')
    print('   Or use: from google.colab import output; output.serve_kernel_port_as_window(8501)')

In [ ]:
#@title 🔄 Alternative: Open directly in Colab (no tunnel needed)
from google.colab import output
output.serve_kernel_port_as_window(8501)